# Washington VRDB Explorer

Queries the SQLite database built by `vrdb_export.py`.

**This notebook reads regulated voter data.** Under RCW 29A.08.720 it may be
used for political purposes but never for commercial advertising or
solicitation; misuse is a class C felony under RCW 29A.08.740. Do not commit
this notebook with output cells populated, and do not commit the database.

Set `EXTRACT_FOLDER` and `EXTRACT_DATE` below to your extract; every path
in this notebook is derived from those two.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

# Point these two at your own extract; every path below is derived from them,
# so a new month is one edit. They are deliberately relative to your home
# directory: the path to a VRDB extract is itself operational detail, and this
# notebook is tracked in a public repository.
#
# EXTRACT_DATE is the date vrdb_export.py named its outputs from; it prints
# them in the run summary.
EXTRACT_FOLDER = Path.home() / "08.2026.WA"
EXTRACT_DATE = "20260803"

DB_PATH = EXTRACT_FOLDER / f"vrdb_{EXTRACT_DATE}.sqlite"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def q(sql, **params):
    return pd.read_sql_query(sql, conn, params=params)


q("""
  SELECT name, (SELECT COUNT(*) FROM sqlite_master m2
                WHERE m2.type='index' AND m2.tbl_name=m.name) AS indexes
  FROM sqlite_master m WHERE type='table' ORDER BY name
""")

In [ ]:
pd.DataFrame(
    [(t, q(f"SELECT COUNT(*) n FROM {t}").n[0])
     for t in ("voters", "voting_history", "districts_precincts")],
    columns=["table", "rows"],
)

In [ ]:
q("""
  SELECT c.name AS county, COUNT(*) AS voters,
         SUM(v.StatusCode = 'Active') AS active,
         SUM(v.StatusCode = 'Inactive') AS inactive
  FROM voters v JOIN ref_county c ON c.code = v.CountyCode
  GROUP BY c.name ORDER BY voters DESC
""")

In [ ]:
ages = q("""
  SELECT (Birthyear / 10) * 10 AS decade, COUNT(*) AS voters
  FROM voters WHERE Birthyear IS NOT NULL
  GROUP BY decade ORDER BY decade
""")
ages.plot.bar(x="decade", y="voters", legend=False, title="Voters by birth decade")

In [ ]:
q("""
  SELECT h.ElectionDate,
         COUNT(*) AS ballots,
         ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM voters), 1) AS pct_of_extract
  FROM voting_history h
  GROUP BY h.ElectionDate ORDER BY h.ElectionDate
""")

In [ ]:
q("""
  SELECT c.name AS county, COUNT(*) AS never_voted_2025_general
  FROM voters v JOIN ref_county c ON c.code = v.CountyCode
  WHERE v.StatusCode = 'Active' AND NOT EXISTS (
      SELECT 1 FROM voting_history h
      WHERE h.StateVoterID = v.StateVoterID AND h.ElectionDate = '2025-11-04'
  )
  GROUP BY c.name ORDER BY never_voted_2025_general DESC LIMIT 15
""")

## Filter voters to one district

This replaces the manual procedure in `Filtering the VRDB in Excel.pdf`: no
dropdown, no copy-paste. The join is on the literal `(CountyCode, PrecinctPart)`
string, which matches 99.999% of voters. Never cast `PrecinctPart` to a number:
the format varies by county (`311.1`, `0600.919`, `2401.A`, `175-3`, `12-A`,
`202-01`, `2169`) and casting would destroy leading zeros and merge distinct
precincts across counties.

In [ ]:
q("""
  SELECT DistrictType, COUNT(DISTINCT DistrictName) AS districts
  FROM districts_precincts GROUP BY DistrictType ORDER BY districts DESC
""")

In [ ]:
DISTRICT = "FIRE DISTRICT NO. 8"

q("""
  SELECT v.StateVoterID, v.FName, v.LName, v.RegCity, v.PrecinctPart,
         v.StatusCode, v.LastVoted
  FROM voters v
  JOIN districts_precincts d
    ON d.CountyCode = v.CountyCode AND d.PrecinctPart = v.PrecinctPart
  WHERE d.DistrictName = :district AND v.StatusCode = 'Active'
  ORDER BY v.LName, v.FName
  LIMIT 50
""", district=DISTRICT)